# 3. Plot Paper Figures

Compact plotting notebook that centralizes shared styles and result loading.


In [ ]:
from collections import defaultdict
from pathlib import Path
import gc, pickle, time
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
PICKLE_DIR = PROJECT_ROOT / 'data' / 'Openalex_2025' / 'pickles'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'results'
START_YEAR, END_YEAR = 2000, 2025
COUNTRIES = ['US','CN','GB','DE','JP','IT','FR','CA','IN','KR','ES','AU','BR','NL','TR']
COUNTRIES_OT = COUNTRIES + ['OT']
COUNTRIES_TOTAL = COUNTRIES + ['CN_Excluded','Total']
DOMAINS = ['Health Sciences','Life Sciences','Social Sciences','Physical Sciences']
FIELDS = ['Biochemistry, Genetics and Molecular Biology','Decision Sciences','Neuroscience','Veterinary','Immunology and Microbiology','Economics, Econometrics and Finance','Chemical Engineering','Medicine','Agricultural and Biological Sciences','Health Professions','Social sciences','Materials Science','Environmental Science','Earth and Planetary Sciences','Engineering','Dentistry','Physics and Astronomy','Business, Management and Accounting','Psychology','Energy','Arts and Humanities','Computer Science','Chemistry','Mathematics','Nursing','Pharmacology, Toxicology and Pharmaceutics']

def load_pickle(name):
    with open(PICKLE_DIR / name, 'rb') as fh:
        return pickle.load(fh)

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

def load_csr_arrays():
    mat = load_pickle('citation_matrix_csr.pickle')
    indptr, indices = mat.indptr, mat.indices
    del mat; gc.collect()
    return indptr, indices

def load_wos_valid_set():
    wos_issn = set(pd.read_csv(PICKLE_DIR / 'wos2025_Journal_ISSN_all.csv')['ISSN'])
    paper_issn = load_pickle('Paper_ISSN_list.pickle')
    valid = {pid for pid, issns in paper_issn.items() if any(x in wos_issn for x in issns)}
    valid.discard('')
    del wos_issn, paper_issn; gc.collect()
    return valid

def refs(indptr, indices, pid):
    return set(indices[indptr[pid]:indptr[pid+1]])

def norm_country(country, country_to_idx):
    return country if country in country_to_idx else 'OT'

def write_year_country(table, out_path, years, countries):
    ensure_dir(Path(out_path).parent)
    with open(out_path, 'w', encoding='utf-8') as fh:
        fh.write('year\t' + ''.join(f'{c}\t' for c in countries) + '\n')
        for year in years:
            fh.write(f'{year}\t' + ''.join(f'{table[year][c]}\t' for c in countries) + '\n')

def write_array(data, out_path, years, cols):
    ensure_dir(Path(out_path).parent)
    with open(out_path, 'w', encoding='utf-8') as fh:
        fh.write('year\t' + ''.join(f'{c}\t' for c in cols) + '\n')
        for i, year in enumerate(years):
            fh.write(f'{year}\t' + ''.join(f'{data[i,j]}\t' for j in range(len(cols))) + '\n')

def write_matrix_by_year(data, out_dir, prefix, years, countries):
    ensure_dir(out_dir)
    for year in years:
        with open(Path(out_dir)/f'{prefix}_{year}.txt', 'w', encoding='utf-8') as fh:
            fh.write('\\i(\\g(\\ab(d))\\-(i,j))\t' + ''.join(f'{c}\t' for c in countries) + '\n')
            for citing in countries:
                fh.write(f'{citing}\t')
                for cited in countries:
                    value = 0 if citing == cited else data[year][citing][cited]
                    fh.write(f'{value}\t')
                fh.write('\n')


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MultipleLocator, FuncFormatter
from matplotlib.lines import Line2D

COLORS=['#000000','#BE0F2D','#003559','#ff7aa2','#446c37','#2981C0','#4763A7','#5a0303','#9c6644','#d6ce93','#a0db72','#86aadc','#61A179','#5ec2b7','#b8b8ff','#954B93']
MARKERS=['s','o','^','v','<','>','D','d','p','*','H','h','X','|','P','8']
FIG_DIR=PROJECT_ROOT/'outputs'/'figures'; ensure_dir(FIG_DIR)

def savefig(name):
    plt.savefig(FIG_DIR/name,bbox_inches='tight',dpi=600,pad_inches=0.05); plt.show()

def read_result(*parts, sep='\t'):
    return pd.read_csv(OUTPUT_ROOT.joinpath(*parts), sep=sep)

def plot_fig1():
    df=read_result('p1_inter_citation_rate','ICR_of_Country_in_Total_interval10.txt')
    norm=read_result('p1_inter_citation_rate','ICR_norm_of_Country_in_Total_interval10.txt')
    fig,ax=plt.subplots(1,2,figsize=(10,4))
    row=df[df.year==2024].iloc[0]; vals=np.array([row[c] for c in COUNTRIES]); order=np.argsort(vals)
    ax[0].bar(np.array(COUNTRIES)[order], vals[order], color=[COLORS[COUNTRIES.index(c)] for c in np.array(COUNTRIES)[order]])
    ax[0].set_ylabel('$f$'); ax[0].tick_params(axis='x',rotation=45)
    for i,c in enumerate(COUNTRIES): ax[1].plot(norm.year,norm[c],marker=MARKERS[i],color=COLORS[i],label=c)
    ax[1].axhline(1,color='gray',ls='--',lw=.8); ax[1].set_ylabel('$\\tilde f$'); ax[1].legend(ncol=3,fontsize=7,frameon=False)
    savefig('Fig1.pdf')

def plot_figS_f_year():
    df=read_result('p1_inter_citation_rate','ICR_of_Country_in_Total_interval10.txt')
    fig,ax=plt.subplots(figsize=(7,4))
    for i,c in enumerate(COUNTRIES): ax.plot(df.year,df[c],marker=MARKERS[i],color=COLORS[i],label=c)
    ax.set_ylabel('$f$'); ax.legend(ncol=4,fontsize=7,frameon=False); savefig('FigS_f_year.pdf')

def plot_delta_i():
    df=read_result('p3_domestic_referencing_rate_delta','domestic_referencing_rate_relatively_mean_of_Country_in_Total.txt')
    fig,ax=plt.subplots(figsize=(7,4))
    for i,c in enumerate(COUNTRIES): ax.plot(df.year,df[c],marker=MARKERS[i],color=COLORS[i],label=c)
    ax.plot(df.year,df[COUNTRIES].mean(axis=1),color='gray',ls='--',label='Average')
    ax.set_ylabel('$\\Delta_i$'); ax.legend(ncol=4,fontsize=7,frameon=False); savefig('FigS_Delta_i.pdf')

def plot_delta_j_i(q1=False):
    folder='p4_inter_referencing_rate_Q1' if q1 else 'p4_inter_referencing_rate'
    targets=['CN'] if q1 else ['US','CN','GB']
    fig,axes=plt.subplots(1,len(targets),figsize=(5*len(targets),4),squeeze=False)
    for ax,target in zip(axes.ravel(),targets):
        rows=[]
        for y in range(2000,2025):
            df=pd.read_csv(OUTPUT_ROOT/f'{folder}/inter_referencing_rate_delta_matrix_interval10_{y}.txt',sep='\t').set_index('\\i(\\g(\\ab(d))\\-(i,j))')
            rows.append({'year':y, **{c:df[target][c] for c in COUNTRIES if c!=target}})
        d=pd.DataFrame(rows)
        for i,c in enumerate([c for c in COUNTRIES if c!=target]): ax.plot(d.year,d[c],marker=MARKERS[i],color=COLORS[i],label=c)
        ax.set_title(target); ax.set_ylabel('$\\Delta_{j,i}$')
    axes[0,0].legend(ncol=3,fontsize=7,frameon=False); savefig('FigS_Delta_j_i_Q1.pdf' if q1 else 'FigS_Delta_j_i.pdf')

def plot_figS_f_norm_field_corr():
    fields=['Health Sciences','Life Sciences','Social Sciences','Physical Sciences']
    fig,axes=plt.subplots(2,2,figsize=(10,7),sharex=True)
    for ax,field in zip(axes.ravel(),fields):
        df=read_result('p1_inter_citation_rate_corr',f'ICR_norm_of_Country_in_{field}_interval10.txt')
        for i,c in enumerate(COUNTRIES): ax.plot(df.year,df[c],marker=MARKERS[i],color=COLORS[i],label=c)
        ax.axhline(1,color='gray',ls='--',lw=.8); ax.set_title(field)
    axes[0,0].legend(ncol=3,fontsize=7,frameon=False); savefig('FigS_f_norm_field_corr.pdf')

def plot_reference_growth_ratio():
    def transform(count_path,rate_path):
        count=pd.read_csv(count_path,sep='\t'); rate=pd.read_csv(rate_path,sep='\t'); count=count[count.year!=2025].copy(); rc=count.copy(); rr=rate.copy()
        for c in COUNTRIES:
            rate[c]=((1+rr['Total']-rr[c]*rc[c]/rc['Total'])/(1+rr[c]))**0.5
            count[c]=rc[c]/(rc['Total']-rc[c])
        return count,rate
    pc,pr=transform(OUTPUT_ROOT/'p5_paper_count_modify_bootstrap_simple/Paper_Count_of_Country.txt',OUTPUT_ROOT/'p5_paper_rate_modify_bootstrap_simple/Paper_Growth_Rate_of_Country.txt')
    rc,rr=transform(OUTPUT_ROOT/'p6_reference_list_length/Country_reference_count_R_L.txt',OUTPUT_ROOT/'p6_reference_list_length/Country_reference_R_L_growth_rate.txt')
    years=list(range(2000,2025)); fig,ax=plt.subplots(1,2,figsize=(12,5),sharex=True)
    for i,c in enumerate(COUNTRIES):
        ax[0].plot(years,rc[c]-rr[c],marker=MARKERS[i],color=COLORS[i],label=c)
        ax[1].plot(years,pc[c]-pr[c],marker=MARKERS[i],color=COLORS[i],label=c)
    ax[0].set_title('Reference volume'); ax[1].set_title('Publication volume')
    for a in ax: a.axhline(0,color='gray',ls='--',lw=.8)
    ax[0].legend(ncol=4,fontsize=7,frameon=False); savefig('FigS_reference_growth_rate_ratio_2panel_R_L.pdf')


In [ ]:
# plot_fig1()
# plot_figS_f_year()
# plot_figS_f_norm_field_corr()
# plot_delta_i()
# plot_delta_j_i(q1=False)
# plot_delta_j_i(q1=True)
# plot_reference_growth_ratio()


## Manual visual-check note

`Fig2.py`, `Fig3_paper_2X2.py`, China/India stress-test figures, and the reference-growth dot merge figure contain many final visual-layout decisions. Their upstream data logic is covered by notebooks 0 and 1; exact final PDF composition should be compared manually with submitted figures before replacing the original figure scripts.